[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509-notebooks/blob/main/Module1/3_AllTheModels_Classification.ipynb)

# Module 1.3 - All the Models (Classification)

**OPIM 5509: Introduction to Deep Learning - University of Connecticut**

Same data. Same skeleton. Different question.

Instead of "what is this block group worth?" we ask "**is this block group above the state median?**" - a yes/no question. That changes three things and only three things:

1. The **target** gets recoded to 0 or 1
2. The **models** switch to their classifier versions
3. The **metrics** change - accuracy, precision, recall, a confusion matrix, and AUC replace MAE and R2

Everything else - the split, the scaler, the fit, the predict - is copied straight from notebook 2.

🔷 **The nugget:** a classifier does not really output a class. It outputs a **probability**, and then a threshold turns that into a class. Whoever picks the threshold is making a business decision, not a technical one.

🔴
<!-- 🎙 DAVE TALKING POINTS (invisible when rendered - double-click to read):
Video 7 - Classification - a different question, the same skeleton

- OPEN by putting notebook 2 side by side with this one on screen. Say: "watch how little changes."
  The whole pedagogical point of this video is the SAMENESS.
- The recode: np.where(y > median, 1, 0). One line. Show value_counts() and point out we get a balanced
  50/50 split BY CONSTRUCTION because we split at the median. Then immediately say: real problems are
  almost NEVER balanced - fraud is 0.1%, churn is 5% - and warn that accuracy is a terrible metric when
  classes are imbalanced. Set up the accuracy-paradox story now, pay it off at the confusion matrix.
- Speed through split + scale. They just saw it. Just say "identical to last notebook" and move.
- Fit the same five model families, now the Classifier versions. Point at the import line - literally
  Regressor -> Classifier.
- CONFUSION MATRIX is the heart of the video. Go slow. Walk all four quadrants out loud in the language
  of THIS problem:
    * true positive  = expensive block group, we said expensive
    * false positive = cheap block group, we said expensive   (a wasted marketing dollar)
    * false negative = expensive block group, we said cheap   (a missed opportunity)
  Give the medical-test analogy too - it always lands.
- Precision vs recall: precision = "when I said yes, how often was I right." recall = "of all the real
  yeses, how many did I catch." Then the punchline: you can always trade one for the other by moving the
  threshold, so ASK WHICH ERROR COSTS MORE before you optimize anything.
- predict_proba is the BRIDGE TO MODULE 2 - say this explicitly and slowly. Show the raw probabilities.
  Show that predict() is just predict_proba() > 0.5. Then say: "in two weeks that probability comes out of
  a SIGMOID at the end of a neural network, and the decision you make with it is exactly the same."
  This is the single most important sentence in Module 1.
- Show the threshold sweep. Watch precision and recall trade off as the threshold moves. Land the point
  that 0.5 is a DEFAULT, not a law.
- ROC/AUC briefly: AUC summarizes performance across ALL thresholds, so it is threshold-free.
  0.5 is coin-flipping, 1.0 is perfect.
- CLOSE Module 1: "you now own the skeleton for regression AND classification. Module 2 swaps in a neural
  network and changes nothing else. Go do Assignment 1."
-->

## 1. Setup - identical to notebook 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# the SAME model families - just the Classifier versions
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

# classification metrics replace MAE / RMSE / R2
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve, precision_score, recall_score)

from sklearn.datasets import fetch_california_housing

df = fetch_california_housing(as_frame=True).frame
df = df[(df["AveOccup"] <= 10) & (df["AveRooms"] <= 20)].copy()

print("Shape:", df.shape)

## 2. Recode the target

We are making up our own question: **is this block group above the statewide median value?**

One line of `np.where` turns a continuous target into a binary one.

In [ ]:
median_value = df["MedHouseVal"].median()

df["EXPENSIVE"] = np.where(df["MedHouseVal"] > median_value, 1, 0)

print(f"Splitting at the median: {median_value:.3f}  (${median_value * 100_000:,.0f})")
print()
print(df["EXPENSIVE"].value_counts())
print()
print("Class balance:")
print(df["EXPENSIVE"].value_counts(normalize=True).round(3))

A near-perfect 50/50 split - **by construction**, because we cut at the median.

**Caution:** real classification problems are almost never balanced. Fraud might be 0.1% of transactions; churn might be 5% of customers. When classes are lopsided, **accuracy becomes a useless metric** - a model that predicts "not fraud" every single time scores 99.9% accuracy and is worth nothing. Keep that in mind when we reach the confusion matrix.

## 3. Split and scale - copied straight from notebook 2

Nothing new here. Same split, same leakage discipline: `fit_transform` on train, `transform` on test.

In [ ]:
y = df["EXPENSIVE"]

# Drop the binary target AND the continuous value it came from -
# leaving MedHouseVal in X would hand the model the answer directly.
X = df.drop(columns=["EXPENSIVE", "MedHouseVal"])

feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, shuffle=True, random_state=42, stratify=y
)

X_train = np.array(X_train); X_test = np.array(X_test)
y_train = np.array(y_train); y_test = np.array(y_test)

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)   # LEARN from train
X_test = scaler.transform(X_test)         # only APPLY to test

print("X_train:", X_train.shape, "  X_test:", X_test.shape)
print("Train class balance:", np.round(np.bincount(y_train) / len(y_train), 3))

**Remember:** dropping `MedHouseVal` from `X` is not optional. We built `EXPENSIVE` *from* it, so leaving it in would let the model read the answer off the feature list and score a perfect 100%. That is **target leakage**, and it is the most embarrassing way to get a suspiciously great result.

The `stratify=y` argument is new: it keeps the class balance identical in train and test. Cheap insurance, especially when classes are imbalanced.

## 4. Fit the models

Same loop as notebook 2. The only edit is `Regressor` becoming `Classifier`.

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    """Fit a classifier and return its train/test scores."""
    model.fit(X_tr, y_tr)
    tr_pred = model.predict(X_tr)
    te_pred = model.predict(X_te)
    te_prob = model.predict_proba(X_te)[:, 1]      # probability of class 1
    return {
        "Model": name,
        "Train Acc": accuracy_score(y_tr, tr_pred),
        "Test Acc": accuracy_score(y_te, te_pred),
        "Precision": precision_score(y_te, te_pred),
        "Recall": recall_score(y_te, te_pred),
        "AUC": roc_auc_score(y_te, te_prob),
    }


models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree":       DecisionTreeClassifier(random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting":   GradientBoostingClassifier(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=10),
}

rows = []
for name, model in models.items():
    print("Fitting", name, "...")
    rows.append(evaluate(name, model, X_train, y_train, X_test, y_test))

results = pd.DataFrame(rows).set_index("Model").round(4)
print("\nDone.")
results

## 5. The confusion matrix

Accuracy is one number, and one number hides everything interesting. The confusion matrix shows you **which kind** of mistake the model is making - and the two kinds usually cost very different amounts.

In [ ]:
best_name = results["AUC"].idxmax()
best_model = models[best_name]
best_pred = best_model.predict(X_test)
best_prob = best_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt=",d", cmap="YlOrBr", cbar=False,
            xticklabels=["predicted cheap", "predicted expensive"],
            yticklabels=["actually cheap", "actually expensive"])
plt.title(f"Confusion matrix - {best_name}")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True negatives  {tn:>6,}   cheap, called cheap          - correct")
print(f"False positives {fp:>6,}   cheap, called expensive      - a wasted marketing dollar")
print(f"False negatives {fn:>6,}   expensive, called cheap      - a missed opportunity")
print(f"True positives  {tp:>6,}   expensive, called expensive  - correct")

Read those four quadrants in the language of the actual problem, not in the language of statistics:

- A **false positive** means you spent money marketing to a neighborhood that could not afford you.
- A **false negative** means you skipped a neighborhood that could have.

Which one hurts more? That is a **business** question, and the answer determines everything below.

In [ ]:
print(classification_report(y_test, best_pred,
                            target_names=["cheap (0)", "expensive (1)"]))

| Metric | The question it answers |
| --- | --- |
| **Precision** | When the model said "expensive," how often was it right? |
| **Recall** | Of all the genuinely expensive block groups, how many did we catch? |
| **F1** | The harmonic mean of the two - one number when you must have one number |

You can always trade one for the other by moving the decision threshold. So decide **which error costs more** before you start optimizing anything.

## 6. Probabilities, thresholds, and the bridge to Module 2

Here is the idea that carries the rest of the semester.

A classifier does not really output a class. It outputs a **probability**. `predict()` is just `predict_proba()` compared against 0.5 - and 0.5 is a **default**, not a law of nature.

In [ ]:
# The raw probabilities the model actually produces
probs = best_model.predict_proba(X_test)[:, 1]

peek = pd.DataFrame({
    "P(expensive)": probs[:10].round(3),
    "predicted class": best_pred[:10],
    "actual class": y_test[:10],
})
peek

In [ ]:
# Proof that predict() is just a threshold at 0.5
manual = (probs > 0.5).astype(int)
print("Manual threshold matches .predict() exactly:", np.array_equal(manual, best_pred))

In [ ]:
# Move the threshold and watch precision and recall trade places.
rows = []
for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    pred_t = (probs > t).astype(int)
    rows.append({
        "threshold": t,
        "accuracy": accuracy_score(y_test, pred_t),
        "precision": precision_score(y_test, pred_t, zero_division=0),
        "recall": recall_score(y_test, pred_t),
    })

pd.DataFrame(rows).set_index("threshold").round(3)

Raise the threshold and precision climbs while recall falls - the model gets pickier, so it is right more often when it speaks up, but it stays quiet more often. Lower it and the trade runs the other way.

🔷 **The nugget:** in two weeks that probability will come out of a **sigmoid** at the end of a neural network. The number arrives from a completely different machine, and what you do with it - pick a threshold, read a confusion matrix, weigh false positives against false negatives - does not change at all.

## 7. ROC and AUC

If the threshold is a choice, how do you compare models without committing to one? You sweep across *every* threshold and measure the area under the resulting curve.

`AUC = 0.5` is a coin flip. `AUC = 1.0` is perfect.

In [ ]:
plt.figure(figsize=(6.5, 6))

for name, model in models.items():
    p = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, p)
    plt.plot(fpr, tpr, linewidth=1.6, label=f"{name} (AUC={roc_auc_score(y_test, p):.3f})")

plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="coin flip (AUC=0.500)")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curves - every threshold at once")
plt.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

## 8. Module 1, in one picture

```
                    REGRESSION                    CLASSIFICATION
  read data         fetch_california_housing()    fetch_california_housing()
  split             train_test_split()            train_test_split()
  scale             fit_transform / transform     fit_transform / transform
  ---------------------------------------------------------------------------
  model             RandomForestRegressor()       RandomForestClassifier()
  ---------------------------------------------------------------------------
  fit               .fit(X_train, y_train)        .fit(X_train, y_train)
  predict           .predict(X_test)              .predict(X_test)
  evaluate          MAE, RMSE, R2                 accuracy, precision, recall, AUC
                    predicted-vs-actual plot      confusion matrix, ROC
```

Only the middle line changed. In Module 2 that middle line becomes a neural network, and the rows above and below it stay exactly where they are.

## What you should have after this notebook

- You can turn a regression problem into a classification problem with one `np.where`
- You can explain **target leakage** and say why `MedHouseVal` had to leave `X`
- You can read all four quadrants of a confusion matrix in the language of the business problem
- You can define precision and recall without hedging, and say which one your problem cares about
- You know `predict()` is just `predict_proba()` at a threshold, and that the threshold is yours to choose
- You can interpret an AUC and say why it is threshold-free

**On your own:** rebuild `EXPENSIVE` using the **90th percentile** instead of the median. Now only 10% of block groups are positive. Re-run the notebook. What happens to accuracy, and why is it now a misleading number?

---

**Next:** `Assignment1_OPIM5509.ipynb` - your turn, on data you have not seen.